# all_tables.ipynb

The purpose of this notebook is to investigate whether all tables (with the exception of c_a) may be naively joined to hd. The notebook outputs a .csv file for each table in the code block below. Later blocks output flattened & pivoted tables for ones that do not strictly conform to one row per instutition.

In [7]:
import duckdb
from pathlib import Path
import pandas as pd

current_script_parent = Path.cwd().parent
con = duckdb.connect(current_script_parent / "ipeds.duckdb")

all_tables = ["hd", "ic", "ic_ay", "ic_py", "adm", "efia", "effy", 
              "ef_a", "ef_b", "ef_c", "ef_d", "sfa", "gr", "gr200", 
              "om", "eap", "sal_is", "al", "f1a", "f2", "f3"]

for table in all_tables:
    print(f"Processing table: {table}")

    df = con.execute(f"SELECT * FROM {table} WHERE year = 2023").fetchdf()

    print(f"Saving table: {table} with shape {df.shape}")

    df.to_csv(current_script_parent / "experiments" / "data" / "original" / f"ipeds_{table}_2023.csv", index=False)


Processing table: hd
Saving table: hd with shape (6163, 208)
Processing table: ic
Saving table: ic with shape (6049, 250)
Processing table: ic_ay
Saving table: ic_ay with shape (3825, 294)
Processing table: ic_py
Saving table: ic_py with shape (2141, 134)
Processing table: adm
Saving table: adm with shape (1972, 110)
Processing table: efia
Saving table: efia with shape (5959, 19)
Processing table: effy
Saving table: effy with shape (116431, 151)
Processing table: ef_a
Saving table: ef_a with shape (115156, 156)
Processing table: ef_b
Saving table: ef_b with shape (158917, 24)
Processing table: ef_c
Saving table: ef_c with shape (40395, 8)
Processing table: ef_d
Saving table: ef_d with shape (5646, 47)
Processing table: sfa
Saving table: sfa with shape (5653, 712)
Processing table: gr
Saving table: gr with shape (51368, 145)
Processing table: gr200
Saving table: gr200 with shape (4954, 54)
Processing table: om
Saving table: om with shape (47342, 68)
Processing table: eap
Saving table: e

So it seems that ic, ic_ay, ic_py, adm, efia, ef_d, sfa, gr200, al, f1a, f2, f3 may be naively joined.

effy, ef_a, ef_b, ef_c, gr, om, eap, sal_is will reqiure some pivoting.

In [ ]:
"""
# effy
12-month unduplicated headcount by race/ethnicity, gender and level of student.
"""

df = con.execute(f"SELECT * FROM effy WHERE year = 2023").fetchdf()

# remove year column
df = df.drop(columns=['year'])

# remove cols with all NaN values
df = df.dropna(axis=1, how='all')

# remove cols that start with 'x'
df = df.drop(columns=[col for col in df.columns if col.startswith('x')])

exclude_cols = {'unitid', 'effyalev', 'effylev', 'lstudy'}
demographic_cols = [col for col in df.columns if col not in exclude_cols]

pivot_df = df.pivot(
    index='unitid',
    columns='effyalev',
    values=demographic_cols
)

print(pivot_df.columns)

# Flatten the MultiIndex columns (e.g., ('EFYTOTLT', 1) becomes 'EFFYALEV_1_EFYTOTLT')
pivot_df.columns = [f"effyalev_{col[1]}_{col[0]}" for col in pivot_df.columns]

# Bring UNITID back from the index into a standard column
pivot_df = pivot_df.reset_index()

# replace all '.' with NaN
pivot_df = pivot_df.replace('.', pd.NA)

# export the flattened DataFrame to a CSV file
pivot_df.to_csv(current_script_parent / "experiments" / "data" / "ipeds_effy_2023.csv", index=False)

MultiIndex([('efytotlt',  1.0),
            ('efytotlt',  2.0),
            ('efytotlt',  3.0),
            ('efytotlt',  4.0),
            ('efytotlt',  5.0),
            ('efytotlt', 11.0),
            ('efytotlt', 12.0),
            ('efytotlt', 19.0),
            ('efytotlt', 20.0),
            ('efytotlt', 21.0),
            ...
            ( 'efygukn', 40.0),
            ( 'efygukn', 41.0),
            ( 'efygukn', 42.0),
            ( 'efygukn', 43.0),
            ( 'efygukn', 44.0),
            ( 'efygukn', 45.0),
            ( 'efygukn', 51.0),
            ( 'efygukn', 52.0),
            ( 'efygukn', 59.0),
            ( 'efygukn', 60.0)],
           names=[None, 'effyalev'], length=918)


In [ ]:
"""
ef_a
Race/ethnicity, gender, attendance status, and level of student

Not sure how this is different from effy?
"""

df = con.execute(f"SELECT * FROM effy WHERE year = 2023").fetchdf()

# remove year column
df = df.drop(columns=['year'])

# remove cols with all NaN values
df = df.dropna(axis=1, how='all')

# remove cols that start with 'x'
df = df.drop(columns=[col for col in df.columns if col.startswith('x')])

exclude_cols = {'unitid', 'efalevel', 'line', 'section', 'lstudy'}
demographic_cols = [col for col in df.columns if col not in exclude_cols]

pivot_df = df.pivot(
    index='unitid',
    columns='efalevel',
    values=demographic_cols
)

print(pivot_df.columns)

# Flatten the MultiIndex columns (e.g., ('EFYTOTLT', 1) becomes 'EFFYALEV_1_EFYTOTLT')
pivot_df.columns = [f"effyalev_{col[1]}_{col[0]}" for col in pivot_df.columns]

# Bring UNITID back from the index into a standard column
pivot_df = pivot_df.reset_index()

# replace all '.' with NaN
pivot_df = pivot_df.replace('.', pd.NA)

# export the flattened DataFrame to a CSV file
pivot_df.to_csv(current_script_parent / "experiments" / "data" / "ipeds_effy_2023.csv", index=False)

MultiIndex([('efytotlt',  1.0),
            ('efytotlt',  2.0),
            ('efytotlt',  3.0),
            ('efytotlt',  4.0),
            ('efytotlt',  5.0),
            ('efytotlt', 11.0),
            ('efytotlt', 12.0),
            ('efytotlt', 19.0),
            ('efytotlt', 20.0),
            ('efytotlt', 21.0),
            ...
            ( 'efygukn', 40.0),
            ( 'efygukn', 41.0),
            ( 'efygukn', 42.0),
            ( 'efygukn', 43.0),
            ( 'efygukn', 44.0),
            ( 'efygukn', 45.0),
            ( 'efygukn', 51.0),
            ( 'efygukn', 52.0),
            ( 'efygukn', 59.0),
            ( 'efygukn', 60.0)],
           names=[None, 'effyalev'], length=918)


In [ ]:
"""
ef_b

Age category, gender, attendance status, and level of student.
"""

df = con.execute(f"SELECT * FROM ef_b WHERE year = 2023").fetchdf()

# remove year column
df = df.drop(columns=['year'])

# remove cols with all NaN values
df = df.dropna(axis=1, how='all')

# remove cols that start with 'x'
df = df.drop(columns=[col for col in df.columns if col.startswith('x')])

exclude_cols = {'unitid', 'efbage', 'line', 'lstudy'}
demographic_cols = [col for col in df.columns if col not in exclude_cols]

pivot_df = df.pivot(
    index='unitid',
    columns=['efbage', 'lstudy'],
    values=demographic_cols
)

print(pivot_df.columns)

# Flatten the MultiIndex columns (e.g., ('EFYTOTLT', 1) becomes 'EFFYALEV_1_EFYTOTLT')
pivot_df.columns = [f"efbage_{col[2]}_{col[1]}_{col[0]}" for col in pivot_df.columns]

# Bring UNITID back from the index into a standard column
pivot_df = pivot_df.reset_index()

# replace all '.' with NaN
pivot_df = pivot_df.replace('.', pd.NA)

# export the flattened DataFrame to a CSV file
pivot_df.to_csv(current_script_parent / "experiments" / "data" / "ipeds_ef_b_2023.csv", index=False)

MultiIndex([('efage01',  1.0, 1.0),
            ('efage01',  1.0, 2.0),
            ('efage01',  1.0, 5.0),
            ('efage01',  2.0, 1.0),
            ('efage01',  2.0, 2.0),
            ('efage01',  2.0, 5.0),
            ('efage01',  3.0, 1.0),
            ('efage01',  3.0, 2.0),
            ('efage01',  4.0, 1.0),
            ('efage01',  4.0, 2.0),
            ...
            ('efage09', 12.0, 2.0),
            ('efage09', 12.0, 5.0),
            ('efage09', 13.0, 1.0),
            ('efage09', 13.0, 5.0),
            ('efage09', 14.0, 1.0),
            ('efage09', 14.0, 2.0),
            ('efage09', 14.0, 5.0),
            ('efage09',  4.0, 5.0),
            ('efage09', 13.0, 2.0),
            ('efage09',  3.0, 5.0)],
           names=[None, 'efbage', 'lstudy'], length=378)


In [29]:
"""
ef_c

Residence and migration of first-time freshman.
"""

df = con.execute(f"SELECT * FROM ef_c WHERE year = 2023").fetchdf()

# remove year column
df = df.drop(columns=['year'])

# remove cols with all NaN values
df = df.dropna(axis=1, how='all')

# remove cols that start with 'x'
df = df.drop(columns=[col for col in df.columns if col.startswith('x')])

exclude_cols = {'unitid', 'efcstate', 'line'}
demographic_cols = [col for col in df.columns if col not in exclude_cols]

pivot_df = df.pivot(
    index='unitid',
    columns='efcstate',
    values=demographic_cols
)

print(pivot_df.columns)

# Flatten the MultiIndex columns (e.g., ('EFYTOTLT', 1) becomes 'EFFYALEV_1_EFYTOTLT')
pivot_df.columns = [f"efcstate_{col[1]}_{col[0]}" for col in pivot_df.columns]

# Bring UNITID back from the index into a standard column
pivot_df = pivot_df.reset_index()

# replace all '.' with NaN
pivot_df = pivot_df.replace('.', pd.NA)

# export the flattened DataFrame to a CSV file
pivot_df.to_csv(current_script_parent / "experiments" / "data" / "ipeds_ef_c_2023.csv", index=False)

MultiIndex([('efres01',  1.0),
            ('efres01',  2.0),
            ('efres01',  4.0),
            ('efres01',  5.0),
            ('efres01',  6.0),
            ('efres01',  8.0),
            ('efres01',  9.0),
            ('efres01', 10.0),
            ('efres01', 11.0),
            ('efres01', 12.0),
            ...
            ('efres02', 66.0),
            ('efres02', 68.0),
            ('efres02', 69.0),
            ('efres02', 70.0),
            ('efres02', 72.0),
            ('efres02', 78.0),
            ('efres02', 89.0),
            ('efres02', 90.0),
            ('efres02', 98.0),
            ('efres02', 99.0)],
           names=[None, 'efcstate'], length=130)


In [30]:
"""
gr

Graduation rate data, 150% of normal time to complete - cohort year 2018 (4-year) and cohort year 2021 (2-year) institutions
"""

df = con.execute(f"SELECT * FROM gr WHERE year = 2023").fetchdf()

df = df.drop(columns=['year'])
df = df.dropna(axis=1, how='all')
df = df.drop(columns=[col for col in df.columns if col.startswith('x')])

exclude_cols = {'unitid', 'grtype', 'chrtstat',	'section', 'cohort', 'line'}
demographic_cols = [col for col in df.columns if col not in exclude_cols]

pivot_df = df.pivot(
    index='unitid',
    columns='grtype',
    values=demographic_cols
)
print(pivot_df.columns)

pivot_df.columns = [f"grtype_{col[1]}_{col[0]}" for col in pivot_df.columns]
pivot_df = pivot_df.reset_index()
pivot_df = pivot_df.replace('.', pd.NA)
pivot_df.to_csv(current_script_parent / "experiments" / "data" / "ipeds_gr_2023.csv", index=False)

MultiIndex([('grtotlt',  1.0),
            ('grtotlt',  2.0),
            ('grtotlt',  3.0),
            ('grtotlt',  4.0),
            ('grtotlt',  6.0),
            ('grtotlt',  7.0),
            ('grtotlt',  8.0),
            ('grtotlt',  9.0),
            ('grtotlt', 10.0),
            ('grtotlt', 11.0),
            ...
            ('grnralw', 37.0),
            ('grnralw', 40.0),
            ('grnralw', 41.0),
            ('grnralw', 42.0),
            ('grnralw', 43.0),
            ('grnralw', 44.0),
            ('grnralw', 45.0),
            ('grnralw', 46.0),
            ('grnralw', 47.0),
            ('grnralw', 48.0)],
           names=[None, 'grtype'], length=1260)


In [33]:
"""
om

Award and enrollment data at four, six and eight years of entering degree/certificate-seeking undergraduate cohorts from 2016-17 at degree-granting institutions, by Pell status
"""

df = con.execute(f"SELECT * FROM om WHERE year = 2023").fetchdf()

df = df.drop(columns=['year'])
df = df.dropna(axis=1, how='all')
df = df.drop(columns=[col for col in df.columns if col.startswith('x')])

exclude_cols = {'unitid', 'omchrt'}
demographic_cols = [col for col in df.columns if col not in exclude_cols]

pivot_df = df.pivot(
    index='unitid',
    columns='omchrt',
    values=demographic_cols
)
print(pivot_df.columns)

pivot_df.columns = [f"omchrt_{col[1]}_{col[0]}" for col in pivot_df.columns]
pivot_df = pivot_df.reset_index()
pivot_df = pivot_df.replace('.', pd.NA)
pivot_df.to_csv(current_script_parent / "experiments" / "data" / "ipeds_om_2023.csv", index=False)

MultiIndex([('omrchrt', 10),
            ('omrchrt', 11),
            ('omrchrt', 12),
            ('omrchrt', 20),
            ('omrchrt', 21),
            ('omrchrt', 22),
            ('omrchrt', 30),
            ('omrchrt', 31),
            ('omrchrt', 32),
            ('omrchrt', 40),
            ...
            ('omenrup', 22),
            ('omenrup', 30),
            ('omenrup', 31),
            ('omenrup', 32),
            ('omenrup', 40),
            ('omenrup', 41),
            ('omenrup', 42),
            ('omenrup', 50),
            ('omenrup', 51),
            ('omenrup', 52)],
           names=[None, 'omchrt'], length=390)


In [ ]:
"""
eap

Number of staff by occupational category, faculty and tenure status.

This one is really sparse.
"""

df = con.execute(f"SELECT * FROM eap WHERE year = 2023").fetchdf()

df = df.drop(columns=['year'])
df = df.dropna(axis=1, how='all')
df = df.drop(columns=[col for col in df.columns if col.startswith('x')])

exclude_cols = {'unitid', 'eapcat', 'occupcat', 'facstat', 'xeaptot', 'eaptot'}
demographic_cols = [col for col in df.columns if col not in exclude_cols]

pivot_df = df.pivot(
    index='unitid',
    columns='eapcat',
    values=demographic_cols
)
print(pivot_df.columns)

pivot_df.columns = [f"eapcat_{col[1]}_{col[0]}" for col in pivot_df.columns]
pivot_df = pivot_df.reset_index()
pivot_df = pivot_df.replace('.', pd.NA)
pivot_df.to_csv(current_script_parent / "experiments" / "data" / "ipeds_eap_2023.csv", index=False)

MultiIndex([(  'eaptyp', 10000.0),
            (  'eaptyp', 10010.0),
            (  'eaptyp', 10020.0),
            (  'eaptyp', 10030.0),
            (  'eaptyp', 10040.0),
            (  'eaptyp', 10041.0),
            (  'eaptyp', 10042.0),
            (  'eaptyp', 10043.0),
            (  'eaptyp', 10044.0),
            (  'eaptyp', 10045.0),
            ...
            ('eapptmed', 37000.0),
            ('eapptmed', 37060.0),
            ('eapptmed', 38000.0),
            ('eapptmed', 38060.0),
            ('eapptmed', 39000.0),
            ('eapptmed', 39060.0),
            ('eapptmed', 40000.0),
            ('eapptmed', 41000.0),
            ('eapptmed', 42000.0),
            ('eapptmed', 49000.0)],
           names=[None, 'eapcat'], length=2096)


In [35]:
"""
sal_is

Number and salary outlays for full-time nonmedical instructional staff, by gender, and academic rank.
"""

df = con.execute(f"SELECT * FROM sal_is WHERE year = 2023").fetchdf()

df = df.drop(columns=['year'])
df = df.dropna(axis=1, how='all')
df = df.drop(columns=[col for col in df.columns if col.startswith('x')])

exclude_cols = {'unitid', 'arank'}
demographic_cols = [col for col in df.columns if col not in exclude_cols]

pivot_df = df.pivot(
    index='unitid',
    columns='arank',
    values=demographic_cols
)
print(pivot_df.columns)

pivot_df.columns = [f"arank_{col[1]}_{col[0]}" for col in pivot_df.columns]
pivot_df = pivot_df.reset_index()
pivot_df = pivot_df.replace('.', pd.NA)
pivot_df.to_csv(current_script_parent / "experiments" / "data" / "ipeds_sal_is_2023.csv", index=False)

MultiIndex([('sainstt', 1),
            ('sainstt', 2),
            ('sainstt', 3),
            ('sainstt', 4),
            ('sainstt', 5),
            ('sainstt', 6),
            ('sainstt', 7),
            ('sainstm', 1),
            ('sainstm', 2),
            ('sainstm', 3),
            ...
            ('sa12mam', 5),
            ('sa12mam', 6),
            ('sa12mam', 7),
            ('sa12maw', 1),
            ('sa12maw', 2),
            ('sa12maw', 3),
            ('sa12maw', 4),
            ('sa12maw', 5),
            ('sa12maw', 6),
            ('sa12maw', 7)],
           names=[None, 'arank'], length=378)
